In [6]:
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import STL
from statsmodels.nonparametric.smoothers_lowess import lowess
from scipy.stats import boxcox
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, acf, pacf
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from itertools import product


# ── Config ─────────────────────────────────────────────────────────────────────
PERIOD = 4
N_LAGS = 4
CSV_IN = 'empirical_data1.csv'

# ── Load & reshape ─────────────────────────────────────────────────────────────
db   = pd.read_csv(CSV_IN)

df = (
    db
    .pivot_table(index=['id','years','quarters'], columns='company', values='sales', aggfunc='first')
    .reset_index()
)

# ── Impute missing values (mean of neighbors) ──────────────────────────────────
for column in df.columns:
    for i in range(len(df[column])):
        if pd.isna(df.at[i, column]):
            if i == 0:
                df.at[i, column] = df.at[i + 1, column]
            elif i == len(df) - 1:
                df.at[i, column] = df.at[i - 1, column]
            else:
                df.at[i, column] = np.mean([df.at[i - 1, column], df.at[i + 1, column]])


# ── Locate the row index for a given year / quarter ────────────────────────────
def get_split_index(data, split_year, split_quarter):
    """
    Returns the positional (0-based) row index where years == split_year and
    quarters == split_quarter. This row is the LAST point of the training
    window (inclusive). Everything after it is the forecast/test window.
    """
    mask = (data['years'] == split_year) & (data['quarters'] == split_quarter)
    idx = data.index[mask]
    if len(idx) == 0:
        raise ValueError(f"No row found for {split_year} Q{split_quarter} in the data.")
    return int(idx[0])


# ── Box Cox ────────────────────────────────────────────────────────────────────
def apply_boxcox(y):
    shift = 0
    if np.any(y <= 0):
        shift = abs(np.min(y)) + 1
    y_transformed, _ = boxcox(y + shift)
    return y_transformed

# ── Decomposition ──────────────────────────────────────────────────────────────
def get_decomposition(y, period):
    if period is not None and period > 1:
        stl    = STL(y, period=period, robust=True)
        result = stl.fit()
        T, S, R = result.trend, result.seasonal, result.resid
    else:
        t = np.arange(len(y))
        T = lowess(y, t, frac=0.3, return_sorted=False)
        S = np.zeros(len(y))
        R = y - T
    return T, S, R

# ── Trend & seasonality strength ───────────────────────────────────────────────
def strength_trend_seasonality(y_bc, period):
    T, S, R = get_decomposition(y_bc, period)
    var_R        = np.var(R, ddof=1)
    var_T_plus_R = np.var(T + R, ddof=1)
    var_S_plus_R = np.var(S + R, ddof=1)
    trend_str    = max(0.0, 1 - var_R / var_T_plus_R) if var_T_plus_R != 0 else 0.0
    seasonal_str = max(0.0, 1 - var_R / var_S_plus_R) if var_S_plus_R != 0 else 0.0
    return trend_str, seasonal_str

# ── Curvature ──────────────────────────────────────────────────────────────────
def curvature(y_bc, period, beta2_max_abs=6.698709392927174):
    y_scaled = (y_bc - np.mean(y_bc)) / np.std(y_bc, ddof=1)

    T_comp, _, _ = get_decomposition(y_scaled, period)

    t = np.arange(1, len(T_comp) + 1, dtype=float)
    t_std = (t - np.mean(t)) / np.std(t, ddof=1)

    X = np.column_stack([np.ones(len(t)), t_std, t_std**2])
    Q, _ = np.linalg.qr(X)
    coeffs, _, _, _ = np.linalg.lstsq(Q, T_comp, rcond=None)

    return float(abs(coeffs[2]) / beta2_max_abs)

# ── Spikiness ──────────────────────────────────────────────────────────────────
def spikiness(y_bc, period):
    _, _, R = get_decomposition(y_bc, period)
    n = len(R)
    def relative_disp(r):
        median_abs = np.median(np.abs(r))
        return 0.0 if median_abs == 0 else np.mean(np.abs(r - np.mean(r))) / median_abs
    loo_disp = np.array([relative_disp(np.delete(R, i)) for i in range(n)])
    d_bar = np.mean(loo_disp)
    return 0.0 if d_bar == 0 else float(np.mean(np.abs(loo_disp - d_bar) / d_bar))

# ── Feature extraction per series ─────────────────────────────────────────────
def extract_features(y, period=PERIOD):
    if len(y) < period * 2 + 2:
        return None
    try:
        y_bc = apply_boxcox(y)
        trend, seasonality = strength_trend_seasonality(y_bc, period)
        curv               = curvature(y_bc, period)
        spik               = spikiness(y_bc, period)
        feats = {"trend": trend, "seasonality": seasonality, "curvature": curv, "spikiness": spik}
        if any(not np.isfinite(v) for v in feats.values()):
            return None
        return {k: float(np.clip(v, 0, 1)) for k, v in feats.items()}
    except Exception as e:
        return None

# ── Process one (or several) chosen series ─────────────────────────────────────
# Features are computed ONLY on the TRAIN window (row 0 .. split_idx inclusive),
# i.e. from the first point of the data up to the quarter you provide below —
# never on the full series, and only for the series you asked for (not all 30).
def process_selected_series(data, split_idx, columns, period=PERIOD):
    series_df = data.drop(columns=['id', 'years', 'quarters'])
    train_df  = series_df.iloc[:split_idx + 1]   # inclusive of the split point
    results, skipped = [], []
    for col in columns:
        y = train_df[col].dropna().values.astype(float)
        feats = extract_features(y, period=period)
        if feats is not None:
            results.append({"series": col, **feats})
        else:
            skipped.append(col)
    return pd.DataFrame(results)


In [ ]:
def run_bwm(result_df):
    print("\nBWM Results:")
    feature_cols  = ["trend", "seasonality", "curvature", "spikiness"]
    normalized_df = result_df[["series"] + feature_cols].copy()

    df_raw     = normalized_df.drop(columns=["series"]).reset_index(drop=True)
    series_col = normalized_df["series"].reset_index(drop=True)
    features   = df_raw.columns.tolist()

    records = []
    for i, row in df_raw.iterrows():
        vals    = row.values.astype(float)
        total   = vals.sum()
        weights = vals / total if total > 0 else np.zeros(len(vals))
        record  = {"series": series_col.iloc[i]}
        for f, w_val in zip(features, weights):
            record[f"w_{f}"] = round(float(w_val), 4)
        records.append(record)

    results_df   = pd.DataFrame(records)
    weight_cols  = [c for c in results_df.columns if c.startswith("w_")]
    w            = results_df[weight_cols].values.astype(float)
    series_names = results_df["series"].tolist()
    N_SERIES     = len(series_names)

    base_scores = np.array([
        [0.6549,  0.2406, 0.3401, 0.0274],   # BSTS
        [0.6469, -0.0031, 0.3890, 0.0060],   # ETS
        [0.6480,  0.1742, 0.3787, 0.0293],   # GradientBoosting
        [0.5923,  0.1788, 0.4302, -0.0219],  # RandomForest
        [0.6619,  0.1158, 0.3436, -0.0008]   # SARIMA
    ])



    MODELS = ["BSTS", "ETS", "GradientBoosting", "RandomForest", "SARIMA"]
    name_width = max(len(n) for n in series_names) + 2
    header = f"{'Series':<{name_width}}" + "".join(f"{m:>20}" for m in MODELS) + f"{'Best Model':>22}"

    print(header)

    best_models  = []
    bwm_rankings = {}   # series -> list of models sorted by score, best first

    for s in range(N_SERIES):
        ws              = w[s]
        weighted_scores = base_scores @ ws
        best_idx        = np.argmax(weighted_scores)
        best_model      = MODELS[best_idx]
        best_models.append(best_model)

        # Full ranking of models for this series, highest score first
        ranked_idx = np.argsort(weighted_scores)[::-1]
        bwm_rankings[series_names[s]] = [MODELS[i] for i in ranked_idx]

        row = f"{series_names[s]:<{name_width}}"
        for sc in weighted_scores:
            row += f"{sc:>20.4f}"
        row += f"{best_model:>22}"
        print(row)

    bwm_selection = dict(zip(series_names, best_models))
    return bwm_selection, bwm_rankings

In [8]:

def compute_rmse(actual, predicted):
    actual    = np.array(actual, dtype=float)
    predicted = np.array(predicted, dtype=float)
    if len(actual) == 0: return np.nan
    return float(np.sqrt(np.mean((actual - predicted) ** 2)))

def compute_mape(actual, predicted):
    actual    = np.array(actual, dtype=float)
    predicted = np.array(predicted, dtype=float)
    mask = actual != 0
    if mask.sum() == 0: return np.nan
    return float(np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100)

# ============================================================
# MODELS  — each returns (rmse, pred_array)
# ============================================================
def run_sarima(train_arr, val_arr, period=4):
    try:
        train = pd.Series(train_arr.astype(float))
        try:
            d = 0 if adfuller(train.dropna(), autolag="AIC")[1] < 0.05 else 1
        except:
            d = 1
        try:
            diff_s = train.diff(period).dropna()
            D = 0 if (len(diff_s) > 5 and adfuller(diff_s, autolag="AIC")[1] < 0.05) else 1
        except:
            D = 1
        try:
            T  = len(train)
            ci = 1.96 / np.sqrt(T)
            lm = min(20, T // 2 - 1)
            acf_v  = acf(train.dropna(), nlags=lm, fft=True)
            pacf_v = pacf(train.dropna(), nlags=lm, method="ywmle")
            pmax = min(max((k for k in range(1, len(pacf_v)) if abs(pacf_v[k]) > ci), default=1), 3)
            qmax = min(max((k for k in range(1, len(acf_v))  if abs(acf_v[k])  > ci), default=1), 3)
            Pmax = 1 if any(abs(pacf_v[k]) > ci for k in range(period, len(pacf_v), period)) else 0
            Qmax = 1 if any(abs(acf_v[k])  > ci for k in range(period, len(acf_v),  period)) else 0
        except:
            pmax, qmax, Pmax, Qmax = 1, 1, 1, 1

        best_aic, best_fit = np.inf, None
        for p, q, P, Q in product(range(pmax+1), range(qmax+1),
                                   range(Pmax+1), range(Qmax+1)):
            if p == q == P == Q == 0: continue
            try:
                fit = SARIMAX(train, order=(p, d, q),
                              seasonal_order=(P, D, Q, period),
                              enforce_stationarity=False,
                              enforce_invertibility=False).fit(disp=False)
                all_sig = all(abs(fit.params / fit.bse) > 1.96)
                if all_sig and fit.aic < best_aic:
                    best_aic, best_fit = fit.aic, fit
            except: pass

        if best_fit is None:
            for p, q, P, Q in product(range(pmax+1), range(qmax+1),
                                       range(Pmax+1), range(Qmax+1)):
                if p == q == P == Q == 0: continue
                try:
                    fit = SARIMAX(train, order=(p, d, q),
                                  seasonal_order=(P, D, Q, period),
                                  enforce_stationarity=False,
                                  enforce_invertibility=False).fit(disp=False)
                    if fit.aic < best_aic:
                        best_aic, best_fit = fit.aic, fit
                except: pass

        if best_fit is None: return np.nan, np.full(len(val_arr), np.nan)
        pred = best_fit.get_forecast(steps=len(val_arr)).predicted_mean.clip(lower=0).values
        return compute_rmse(val_arr, pred), pred
    except:
        return np.nan, np.full(len(val_arr), np.nan)


def run_ets(train_arr, val_arr, m=4):
    try:
        y = train_arr.astype(float)
        if len(y) < m * 2: return np.nan, np.full(len(val_arr), np.nan)

        def _ets_fit(y, alpha, beta, gamma, m):
            n = len(y)
            l = np.zeros(n); b = np.zeros(n); s = np.zeros(n + m)
            l[0] = np.mean(y[:m])
            b[0] = (np.mean(y[min(m, n-1):min(2*m, n)]) - np.mean(y[:m])) / m
            for i in range(m): s[i] = y[i] - np.mean(y[:m])
            y_hat = np.zeros(n); y_hat[0] = l[0] + b[0] + s[0]
            for t in range(1, n):
                l_prev = l[t-1]; b_prev = b[t-1]; s_prev = s[t]
                l[t]   = alpha * (y[t] - s_prev) + (1 - alpha) * (l_prev + b_prev)
                b[t]   = beta  * (l[t] - l[t-1]) + (1 - beta)  * b_prev
                s[t+m] = gamma * (y[t] - l[t])   + (1 - gamma) * s_prev
                y_hat[t] = l[t] + b[t] + (s[t+m-m+1] if t+1 < n else 0)
            return y_hat, l, b, s[m:m+n]

        best_sse, best_params = np.inf, (0.3, 0.1, 0.2)
        for a in np.arange(0.05, 0.95, 0.1):
            for b in np.arange(0.05, 0.95, 0.1):
                for g in np.arange(0.05, 0.95, 0.1):
                    try:
                        yh, _, _, _ = _ets_fit(y, a, b, g, m)
                        sse = np.sum((y - yh) ** 2)
                        if sse < best_sse: best_sse = sse; best_params = (a, b, g)
                    except: pass

        a, b, g = best_params
        _, level, trend_c, seasonal = _ets_fit(y, a, b, g, m)

        preds = []
        for step in range(1, len(val_arr) + 1):
            s_idx = -(m - (step % m)) if (step % m) != 0 else -m
            preds.append(level[-1] + step * trend_c[-1] + seasonal[s_idx])
        pred = np.maximum(0, np.array(preds))
        return compute_rmse(val_arr, pred), pred
    except:
        return np.nan, np.full(len(val_arr), np.nan)


def run_bsts(train_arr, val_arr, period=4):
    try:
        train = train_arr.astype(float); test = val_arr.astype(float)
        n = len(train); h = len(test)
        if h == 0 or n < period + 2: return np.nan, np.full(h, np.nan)

        var_y  = max(np.var(np.diff(train), ddof=1) if n > 1 else 1.0, 1e-6)
        slope0 = float(np.mean(np.diff(train[:min(4, n)]))) if n > 1 else 0.0

        def kalman(y, F, H, Q, R, x0, P0):
            d = len(x0); x = x0.copy(); P = P0.copy(); states = np.zeros((len(y), d))
            for t in range(len(y)):
                xp = F @ x; Pp = F @ P @ F.T + Q; S = float(H @ Pp @ H.T) + R
                K = (Pp @ H.T) / S
                x = xp + K.flatten() * (y[t] - float(H @ xp))
                P = (np.eye(d) - np.outer(K.flatten(), H.flatten())) @ Pp
                states[t] = x
            return states

        def kforecast(states, F, H, h):
            x = states[-1].copy(); preds = []
            for _ in range(h): x = F @ x; preds.append(float(H @ x))
            return np.array(preds)

        def inrmse(states, H, y):
            return float(np.sqrt(np.mean((y - np.array([float(H @ s) for s in states])) ** 2)))

        F1 = np.array([[1.]]); H1 = np.array([[1.]])
        Q1 = np.array([[var_y * 0.01]]); R1 = var_y * 0.1
        x01 = np.array([train[0]]); P01 = np.array([[var_y]])

        F2 = np.array([[1., 1.], [0., 1.]]); H2 = np.array([[1., 0.]])
        Q2 = np.diag([var_y * 0.01, var_y * 0.001]); R2 = var_y * 0.1
        x02 = np.array([train[0], slope0]); P02 = np.diag([var_y, var_y * 0.1])

        d3 = 2 + (period - 1); F3 = np.zeros((d3, d3))
        F3[0, 0] = 1.; F3[0, 1] = 1.; F3[1, 1] = 1.; F3[2, 2:] = -1.
        for i in range(3, d3): F3[i, i-1] = 1.
        H3 = np.zeros((1, d3)); H3[0, 0] = 1.; H3[0, 2] = 1.
        Q3 = np.zeros((d3, d3))
        Q3[0, 0] = var_y * 0.01; Q3[1, 1] = var_y * 0.001; Q3[2, 2] = var_y * 0.01
        R3 = var_y * 0.1
        si = np.zeros(period - 1)
        if n >= period:
            for s in range(period - 1):
                v = [train[t] - np.mean(train[:period]) for t in range(s, n, period) if t < n]
                if v: si[s] = float(np.mean(v))
        x03 = np.concatenate([[train[0]], [slope0], si])
        P03 = np.diag([var_y] + [var_y * 0.1] + [var_y * 0.5] * (period - 1))

        F4 = np.array([[1., 1.], [0., 0.8]]); H4 = np.array([[1., 0.]])
        Q4 = np.diag([var_y * 0.01, var_y * 0.001]); R4 = var_y * 0.1
        x04 = np.array([train[0], slope0]); P04 = np.diag([var_y, var_y * 0.1])

        specs = [(F1, H1, Q1, R1, x01, P01), (F2, H2, Q2, R2, x02, P02),
                 (F3, H3, Q3, R3, x03, P03), (F4, H4, Q4, R4, x04, P04)]

        best_r = np.inf; best_st = best_F = best_H = None
        for F, H, Q, R, x0, P0 in specs:
            try:
                st = kalman(train, F, H, Q, R, x0, P0)
                r  = inrmse(st, H, train)
                if r < best_r: best_r = r; best_st = st; best_F = F; best_H = H
            except: continue

        if best_st is None: return np.nan, np.full(h, np.nan)
        pred = np.maximum(0, kforecast(best_st, best_F, best_H, h))

        return compute_rmse(test, pred), pred
    except:
        return np.nan, np.full(len(val_arr), np.nan)


def run_random_forest(train_arr, val_arr, n_lags=4):
    try:
        train = train_arr.astype(float); test = val_arr.astype(float)
        if len(train) <= n_lags or len(test) == 0: return np.nan, np.full(len(test), np.nan)
        X_tr = np.array([train[i-n_lags:i][::-1] for i in range(n_lags, len(train))])
        y_tr = train[n_lags:]
        m = RandomForestRegressor(n_estimators=200, max_features="sqrt",
                                  min_samples_leaf=5, oob_score=True,
                                  random_state=42, n_jobs=-1)
        m.fit(X_tr, y_tr)
        history, preds = list(train), []
        for _ in range(len(test)):
            p = m.predict([np.array(history[-n_lags:])[::-1]])[0]
            p = max(0.0, p)
            preds.append(p); history.append(p)
        pred = np.array(preds)
        return compute_rmse(test, pred), pred
    except:
        return np.nan, np.full(len(val_arr), np.nan)


def run_gradient_boosting(train_arr, val_arr, n_lags=4):
    try:
        train = train_arr.astype(float); test = val_arr.astype(float)
        if len(train) <= n_lags or len(test) == 0: return np.nan, np.full(len(test), np.nan)
        X_tr = np.array([train[i-n_lags:i][::-1] for i in range(n_lags, len(train))])
        y_tr = train[n_lags:]
        m = GradientBoostingRegressor(n_estimators=500, learning_rate=0.05,
                                      max_depth=3, min_samples_leaf=5,
                                      subsample=0.8, loss="squared_error",
                                      random_state=42)
        m.fit(X_tr, y_tr)
        history, preds = list(train), []
        for _ in range(len(test)):
            p = m.predict([np.array(history[-n_lags:])[::-1]])[0]
            p = max(0.0, p)
            preds.append(p); history.append(p)
        pred = np.array(preds)
        return compute_rmse(test, pred), pred
    except:
        return np.nan, np.full(len(val_arr), np.nan)


# ============================================================
# HELPER — run all models on one series
# ============================================================
def evaluate_series(train_arr, val_arr):
    rmse_s, pred_s = run_sarima(train_arr, val_arr, PERIOD)
    rmse_e, pred_e = run_ets(train_arr, val_arr, PERIOD)
    rmse_b, pred_b = run_bsts(train_arr, val_arr, PERIOD)
    rmse_r, pred_r = run_random_forest(train_arr, val_arr, N_LAGS)
    rmse_g, pred_g = run_gradient_boosting(train_arr, val_arr, N_LAGS)

    rmse_dict = {"SARIMA": rmse_s, "ETS": rmse_e, "BSTS": rmse_b,
                 "RandomForest": rmse_r, "GradientBoosting": rmse_g}
    pred_dict = {"SARIMA": pred_s, "ETS": pred_e, "BSTS": pred_b,
                 "RandomForest": pred_r, "GradientBoosting": pred_g}

    mape_dict = {}
    for name, pred in pred_dict.items():
        mape_dict[name] = compute_mape(val_arr, pred) if not np.isnan(rmse_dict[name]) else np.nan

    valid_rmse = {k: v for k, v in rmse_dict.items() if not np.isnan(v)}
    valid_mape = {k: v for k, v in mape_dict.items() if not np.isnan(v)}

    best_rmse = min(valid_rmse, key=valid_rmse.get) if valid_rmse else np.nan
    best_mape = min(valid_mape, key=valid_mape.get) if valid_mape else np.nan

    return rmse_dict, mape_dict, pred_dict, best_rmse, best_mape


# ============================================================
# PART 1 — Evaluate the chosen series
# Uses SPLIT_IDX (train: start -> your chosen quarter, test: next quarter ->
# end of data) instead of a fixed 80/20 split, and runs ONLY on the series
# you picked (not all companies).
# ============================================================
def run_evaluation(data, split_idx, series_name):
    # Filter to non-NaN rows for THIS series, same as notebook 1's plot_forecasts
    mask   = data[series_name].notna()
    series = data.loc[mask, series_name].values.astype(float)

    # Map split_idx (a raw dataframe row position) onto a position inside the
    # NaN-filtered series: count how many valid points exist up to and
    # including that row.
    split_pos = int(mask.iloc[:split_idx + 1].sum()) - 1

    train_arr = series[:split_pos + 1]
    val_arr   = series[split_pos + 1:]

    if len(train_arr) < N_LAGS + 4 or len(val_arr) == 0:
        print(f"  Skipping {series_name} — too short (train={len(train_arr)}, test={len(val_arr)})")
        return pd.DataFrame()

    rmse_dict, mape_dict, _, best_rmse, best_mape = evaluate_series(train_arr, val_arr)

    record = {"sample": series_name}
    for model in ["SARIMA", "ETS", "BSTS", "RandomForest", "GradientBoosting"]:
        r = rmse_dict[model]; m = mape_dict[model]
        record[f"rmse_{model}"] = round(r, 4) if not np.isnan(r) else np.nan
        record[f"mape_{model}"] = round(m, 4) if not np.isnan(m) else np.nan
    record["best_model_rmse"] = best_rmse
    record["best_model_mape"] = best_mape

    eval_df = pd.DataFrame([record])

    print("=" * 70)
    print(f"  {series_name:<40}  best_rmse={best_rmse:<20}  best_mape={best_mape}")
    print("=" * 70)

    return eval_df


In [ ]:
def is_flat_forecast(pred, tol=1e-8):
    """
    Returns True if a forecast array is constant (same value)
    across the whole validation/forecast period.
    """
    pred = np.array(pred, dtype=float)
    valid = pred[~np.isnan(pred)]
    if len(valid) == 0:
        return True
    return np.allclose(valid, valid[0], atol=tol, rtol=0)


def resolve_bwm_model(bwm_model, pred_dict, model_ranking):
    """
    If the BWM-selected model is RandomForest or GradientBoosting AND its
    forecast is flat (constant) across the whole validation period,
    fall through to the next-best model by BWM SCORE ranking (not MAPE)
    that is NOT flat. Otherwise, keep the original BWM selection.

    model_ranking: list of model names for this series, ordered best -> worst
                   by BWM weighted score (from bwm_rankings[sid]).
    """
    FLAT_PRONE_MODELS = {"RandomForest", "GradientBoosting"}

    if bwm_model in FLAT_PRONE_MODELS and bwm_model in pred_dict:
        if is_flat_forecast(pred_dict[bwm_model]):
            for candidate in model_ranking:
                if candidate == bwm_model:
                    continue
                if candidate in pred_dict and not is_flat_forecast(pred_dict[candidate]):
                    return candidate
            return bwm_model  # nothing better found — keep original as last resort

    return bwm_model


def build_future_quarters(last_year, last_quarter, n_periods):
    """
    Generate n_periods of (year, quarter) tuples continuing on
    from (last_year, last_quarter).
    """
    out = []
    y, q = last_year, last_quarter
    for _ in range(n_periods):
        q += 1
        if q > 4:
            q = 1
            y += 1
        out.append((y, q))
    return out


def get_extended_axes(data, sid, end_year, end_quarter):
    """
    Returns (ids_s, year_s, quarter_s, series, end_pos) for series `sid`,
    extended with synthetic future rows (NaN target values) if
    (end_year, end_quarter) falls beyond the last real row in the data.

    end_pos is the position (0-indexed, inclusive) in the EXTENDED arrays
    that corresponds to (end_year, end_quarter).
    """
    mask      = data[sid].notna()
    series    = data.loc[mask, sid].values.astype(float)
    ids_s     = data.loc[mask, "id"].values
    year_s    = data.loc[mask, "years"].values
    quarter_s = data.loc[mask, "quarters"].values

    last_year    = int(year_s[-1])
    last_quarter = int(quarter_s[-1])
    last_id      = ids_s[-1]

    # Does (end_year, end_quarter) already exist among real rows?
    real_match = np.where((year_s == end_year) & (quarter_s == end_quarter))[0]

    if len(real_match) > 0:
        end_pos = int(real_match[0])
        return ids_s, year_s, quarter_s, series, end_pos

    # ── Not in the dataset -> extend forward with synthetic future quarters ──
    if (end_year, end_quarter) < (last_year, last_quarter):
        raise ValueError(
            f"Forecast end {end_year}Q{end_quarter} is before the last "
            f"available data point {last_year}Q{last_quarter}."
        )

    # How many quarters beyond the last real row do we need?
    n_future = 0
    y, q = last_year, last_quarter
    while (y, q) != (end_year, end_quarter):
        q += 1
        if q > 4:
            q = 1
            y += 1
        n_future += 1
        if n_future > 400:  # safety guard against runaway loops
            raise ValueError("Forecast end date too far in the future.")

    future_quarters = build_future_quarters(last_year, last_quarter, n_future)
    future_years     = np.array([y for y, q in future_quarters])
    future_quarters_ = np.array([q for y, q in future_quarters])
    future_ids       = np.array([last_id + i + 1 for i in range(n_future)])
    future_series    = np.full(n_future, np.nan)

    ids_ext      = np.concatenate([ids_s, future_ids])
    year_ext     = np.concatenate([year_s, future_years])
    quarter_ext  = np.concatenate([quarter_s, future_quarters_])
    series_ext   = np.concatenate([series, future_series])

    end_pos = len(series_ext) - 1
    return ids_ext, year_ext, quarter_ext, series_ext, end_pos


def plot_forecasts(data, series_to_plot, bwm_selection, bwm_rankings, split_idx,
                    end_year=None, end_quarter=None):
    fig, axes = plt.subplots(len(series_to_plot), 1, figsize=(14, 6 * len(series_to_plot)))
    if len(series_to_plot) == 1:
        axes = [axes]

    colors  = {"SARIMA": "green", "ETS": "red", "BSTS": "purple",
               "RandomForest": "brown", "GradientBoosting": "pink"}
    markers = {"SARIMA": "o", "ETS": "s", "BSTS": "^",
               "RandomForest": "D", "GradientBoosting": "x"}


    for ax, sid in zip(axes, series_to_plot):
        mask = data[sid].notna()

        if end_year is None or end_quarter is None:
            # Original behaviour: run to the last real row
            series  = data.loc[mask, sid].values.astype(float)
            ids_s   = data.loc[mask, "id"].values
            year_s  = data.loc[mask, "years"].values
            quarter_s = data.loc[mask, "quarters"].values
            end_pos = len(series) - 1
        else:
            # ── NEW: extend beyond the dataset if needed for a future forecast ──
            ids_s, year_s, quarter_s, series, end_pos = get_extended_axes(
                data, sid, end_year, end_quarter
            )

        # Map split_idx (raw row position in the ORIGINAL df) onto a position
        # in the filtered/extended series
        split_pos = int(mask.iloc[:split_idx + 1].sum()) - 1

        train_arr = series[:split_pos + 1]
        val_arr   = series[split_pos + 1: end_pos + 1]

        x_train = ids_s[:split_pos + 1]
        x_val   = ids_s[split_pos + 1: end_pos + 1]

        x_labels = [f"{y},Q{q}" for y, q in zip(year_s, quarter_s)]

        _, _, pred_dict, best_rmse, best_mape = evaluate_series(train_arr, val_arr)

        # MAPE can only be computed where we actually have real (non-NaN) actuals


        ax.plot(x_train, train_arr, label="Train", color="blue", linewidth=2)
        # Only plot validation actuals where they're real (not NaN placeholders)
        real_val_mask = ~np.isnan(val_arr)
        if real_val_mask.any():
            ax.plot(x_val[real_val_mask], val_arr[real_val_mask],
                    label="Validation", color="blue", linewidth=2)

        for model, pred in pred_dict.items():
            if not np.all(np.isnan(pred)):
                pred_slice = pred[: len(x_val)]
                ax.plot(x_val, pred_slice, label=model, linestyle="--",
                        color=colors[model], marker=markers[model], markersize=4)

        ax.axvline(x=x_train[-1], color="gray", linestyle=":", alpha=0.7)

        bwm_model      = bwm_selection.get(sid, "N/A")
        model_ranking  = bwm_rankings.get(sid, [])
        bwm_model      = resolve_bwm_model(bwm_model, pred_dict, model_ranking)
        bwm_selection[sid] = bwm_model

        if bwm_model == "N/A":
            print(f"Number of observations is not enough for series '{sid}' "
                  f"(train={len(train_arr)}, val={len(val_arr)}).")

        ax.set_title(f"{sid}  | Chosen model: {bwm_model}",
                     fontsize=12, fontweight="bold")

        ax.set_xlabel("Year/Quarter")
        ax.set_ylabel("Sales in Millions of Units")
        ax.set_xticks(ids_s)
        ax.set_xticklabels(x_labels, rotation=45, ha="right")
        ax.grid(True, alpha=0.3)



    plt.tight_layout(pad=3.0)
    plt.savefig("forecast_plots.png", dpi=150, bbox_inches="tight")
    print("\nSaved: forecast_plots.png")


def save_bwm_forecasts(data, series_to_plot, bwm_selection, split_idx,
                        end_year=None, end_quarter=None, filename="O_bwm_forecasts.csv"):
    records = []
    for sid in series_to_plot:
        mask = data[sid].notna()

        if end_year is None or end_quarter is None:
            series       = data.loc[mask, sid].values.astype(float)
            id_vals      = data.loc[mask, "id"].values
            year_vals    = data.loc[mask, "years"].values
            quarter_vals = data.loc[mask, "quarters"].values
            end_pos = len(series) - 1
        else:
            id_vals, year_vals, quarter_vals, series, end_pos = get_extended_axes(
                data, sid, end_year, end_quarter
            )

        split_pos = int(mask.iloc[:split_idx + 1].sum()) - 1
        train_arr = series[:split_pos + 1]
        val_arr   = series[split_pos + 1: end_pos + 1]

        _, _, pred_dict, _, _ = evaluate_series(train_arr, val_arr)

        bwm_model = bwm_selection.get(sid, None)
        if bwm_model and bwm_model in pred_dict:
            pred = pred_dict[bwm_model]
            for i in range(len(val_arr)):
                records.append({
                    "series": sid,
                    "id": id_vals[split_pos + 1 + i],
                    "year": year_vals[split_pos + 1 + i],
                    "quarter": quarter_vals[split_pos + 1 + i],
                    "actual": val_arr[i],       # NaN for future/unobserved quarters
                    "predicted": pred[i],
                    "model": bwm_model
                })

    forecast_df = pd.DataFrame(records)
    forecast_df.to_csv(filename, index=False)
    print(f"\nSaved BWM forecasts to {filename}")


# ============================================================
# INPUT-DRIVEN RUN
# ============================================================
id_cols = ['id', 'years', 'quarters']
available_series = [c for c in df.columns if c not in id_cols]

def prompt_for_series(available):
    while True:
        name = input(f"Enter the series/company name {available}: ").strip()
        if name in available:
            return name
        print(f"'{name}' not found. Available options: {available}")

def prompt_for_split(data):
    while True:
        try:
            year    = int(input("Enter the YEAR of the last training quarter (e.g. 2017): ").strip())
            quarter = int(input("Enter the QUARTER of the last training quarter (1-4, e.g. 1): ").strip())
            return get_split_index(data, year, quarter), year, quarter
        except ValueError as e:
            print(f"Invalid input ({e}). Please try again.")

# ── UPDATED: no longer requires the end date to exist in the dataset.
#    Any date on/after the training end date is accepted; dates beyond
#    the last real row trigger a genuine future forecast. ──
def prompt_for_forecast_end(data, split_idx, split_year, split_quarter):
    while True:
        try:
            year    = int(input("Enter the YEAR of the last quarter to FORECAST (e.g. 2027): ").strip())
            quarter = int(input("Enter the QUARTER of the last quarter to FORECAST (1-4, e.g. 4): ").strip())
            if (year, quarter) <= (split_year, split_quarter):
                print("The forecast end date must be AFTER the training end date. Please try again.")
                continue
            return year, quarter
        except ValueError as e:
            print(f"Invalid input ({e}). Please try again.")

TARGET_SERIES = prompt_for_series(available_series)
SPLIT_IDX, SPLIT_YEAR, SPLIT_QUARTER = prompt_for_split(df)
END_YEAR, END_QUARTER = prompt_for_forecast_end(df, SPLIT_IDX, SPLIT_YEAR, SPLIT_QUARTER)

print(f"\nSeries        : {TARGET_SERIES}")
print(f"Train window  :  "
      f"({df.loc[0,'years']}Q{df.loc[0,'quarters']}  ->  {df.loc[SPLIT_IDX,'years']}Q{df.loc[SPLIT_IDX,'quarters']})")
print(f"Forecast/test :  "
      f"({df.loc[SPLIT_IDX+1,'years']}Q{df.loc[SPLIT_IDX+1,'quarters']}  ->  {END_YEAR}Q{END_QUARTER})")

# 1) Features — computed only on the training window, only for the chosen series
result_df = process_selected_series(df, SPLIT_IDX, [TARGET_SERIES], period=PERIOD)

REQUIRED_FEATURE_COLS = ["trend", "seasonality", "curvature", "spikiness"]

not_enough_obs = (
    result_df.empty
    or not set(REQUIRED_FEATURE_COLS).issubset(result_df.columns)
    or result_df[REQUIRED_FEATURE_COLS].isna().all(axis=None)
)

if not_enough_obs:
    print(f"\nNumber of observations is not enough ")
else:
    print("\nFeatures preview (computed on the TRAIN window only):")
    print(result_df.to_string(index=False))

    # 2) BWM — picks the best model for the chosen series from those features,
    #    plus the full score-based ranking of all models (best -> worst)
    bwm_selection, bwm_rankings = run_bwm(result_df)

    # 3) Backtest all 5 models on the chosen series over the same split
    eval_df = run_evaluation(df, SPLIT_IDX, TARGET_SERIES)

    # 4) Plot + save the forecast for the chosen series using its BWM-selected model,
    #    extended into the future if END_YEAR/END_QUARTER is beyond the dataset
    series_to_plot = [TARGET_SERIES]
    plot_forecasts(df, series_to_plot, bwm_selection, bwm_rankings, SPLIT_IDX,
                    END_YEAR, END_QUARTER)
    save_bwm_forecasts(df, series_to_plot, bwm_selection, SPLIT_IDX,
                        END_YEAR, END_QUARTER)


Series        : HMD (Nokia)
Train window  :  (2009Q4  ->  2010Q1)
Forecast/test :  (2010Q2  ->  2013Q3)

Number of observations is not enough 
